# Resolving a Lookahead Conflict

The notebook [03-Look-Ahead.ipynb](03-Look-Ahead.ipynb) has discussed the grammar
```
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
```
which is *unambiguous* but nevertheless not an `LALR(1)` grammar:  after an `'X'` has been read and the next
token is a `'U'`, the parser cannot decide whether to reduce with the rule for `b` or with the rule for `c`,
since that is only settled by the token *following* the `'U'`.  `Menhir` therefore reports a *reduce/reduce* conflict
and, with the option `--strict`, refuses to build an `LALR` parser.

The problem is that the grammar tries to interpret an `'X'` either as a `b` or as a `c` depending on the
context.  The remedy is to pull the context `'U'`, which follows both `b` and `c`, into the rules for `b`
and `c`:
```
    a : b "V"
      | c "W"

    b : "X" "U"

    c : "X" "U"
```
This grammar generates the same language, but now the decision between `b` and `c` is postponed until the
decisive token is the *next* token.  We will see that there are no conflicts.

## Specification of the Grammar

In [ ]:
let lexer = {|
{
  open A_parser
}

rule token = parse
  | 'U'                          { U (Parse_tree.token "U" lexbuf) }
  | 'V'                          { V (Parse_tree.token "V" lexbuf) }
  | 'W'                          { W (Parse_tree.token "W" lexbuf) }
  | 'X'                          { X (Parse_tree.token "X" lexbuf) }
  | eof                          { EOF }
  | _                            { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

let grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> U V W X
%token EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> a b c

%%

start: x = a EOF               { x }

a : x = b v = V                { Tree ("a", [x; Token v]) }
  | x = c w = W                { Tree ("a", [x; Token w]) }

b : x = X u = U                { Tree ("b", [Token x; Token u]) }

c : x = X u = U                { Tree ("c", [Token x; Token u]) }
|}

## Specification of the Parser

We create the parser with the option `--strict`, so that `Menhir` would refuse to create a parser if the grammar had a
conflict.  Since **no** warning is printed by the cell below, the grammar has no conflicts and is therefore an
`LALR(1)` grammar.

The `Lark` version needs the keyword argument `keep_all_tokens=True` in order to keep the anonymous terminals in the
parse tree.  In `Menhir`, the semantic actions given above put the tokens into the tree.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

In [ ]:
generate_parser_with ~options:"--lalr --strict" ~name:"a" ~lexer ~grammar

## Inspecting the LALR States

Because of the option `--dump-resolved`, `Menhir` has written the states of the LALR automaton into the file
`_build/a_parser.automaton.resolved`.  The function `dump_states` is the same one that we have used in
[01-Conflicts.ipynb](01-Conflicts.ipynb): it prints this file.

In [ ]:
let dump_states (name : string) : unit =
  print_string (read_file (Printf.sprintf "_build/%s_parser.automaton.resolved" name));
  flush stdout

In [ ]:
dump_states "a"

The interesting state is the one that contains the two marked rules
```
    b -> X U . [ V ]
    c -> X U . [ W ]
```
This is the state that corresponds to the state which caused the conflict in the previous notebook.  In the
notation of the lecture notes it has the form
$$ \{\; b \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'V'}, \quad
       c \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'W'} \;\} $$
and the follow sets of the two rules, which `Menhir` shows in brackets, are now *disjoint*.  Therefore the action
table of this state offers exactly one reduce action per token:  a `'V'` reduces with the rule for `b`, while a `'W'`
reduces with the rule for `c`.  No other state contains more than one marked rule whose position is at the very end,
so there is no conflict anywhere.

Note that `Menhir` marks the position of the parser inside a rule with the character `.` instead of the bullet
`•` that we use in the lecture notes, and that the end of the input is the token `EOF`.

## Parsing Some Strings

Finally, we check that the parser really distinguishes a `b` from a `c`.

In [ ]:
print_string (pretty (parse_string A_parser.start A_lexer.token "XUV"));;
flush stdout

In [ ]:
print_string (pretty (parse_string A_parser.start A_lexer.token "XUW"));;
flush stdout

A string that is not derivable from `a` is rejected.  There is no need for a function like `p_error`:
the scanner raises the exception `Unexpected_input`, whose fields report the message, the line, the column, and the
position of the error.

In [ ]:
let s = "XUZ" in
try ignore (parse_string A_parser.start A_lexer.token s) with
| Unexpected_input { message; line; column; position } ->
    Printf.printf "%s, at line %d col %d\n\n%s%!" message line column (get_context s position)